<a href="https://colab.research.google.com/github/Navya-Mittal/RAG_Document_Q-A/blob/main/RAG_DocumentChat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG: Document Question Answering Pipeline

A complete **Retrieval-Augmented Generation (RAG)** implementation using LangChain, covering document ingestion through to multi-turn conversational Q&A.

**Pipeline overview:**

```
Documents -> Split -> Embed -> VectorStore
                                    |
User Query -> Retrieve -> LLM -> Answer
```

**Parts covered:**
1. Document Loading
2. Document Splitting
3. Vector Stores and Embeddings
4. Advanced Retrieval
5. Question Answering
6. Conversational Chat with Memory
7. Alternative Stack: FAISS + Nomic Embeddings

**Dependencies (all free):**
- Embeddings: `all-MiniLM-L6-v2` via HuggingFace — runs locally on CPU, no API key required
- LLM: Groq free tier — [create a key here](https://console.groq.com/keys)

## Setup and Installation

In [1]:
!pip install -q langchain langchain-community langchain-groq \
    chromadb sentence-transformers pypdf beautifulsoup4 lark scikit-learn docarray \
    langchain-huggingface faiss-cpu einops

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/1

In [2]:
import os
from getpass import getpass

# Groq API key — free tier, no credit card required: https://console.groq.com/keys
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")
print("API key set.")

Enter your Groq API key: ··········
API key set.


In [3]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import shutil
import time

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq

print("Loading embedding model (approx. 90MB, downloaded once)...")
embedding = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

# LLM configuration
# Available models on Groq free tier:
#   "llama-3.1-8b-instant"     -- fastest, lowest latency
#   "llama-3.3-70b-versatile"  -- highest quality responses
#   "mixtral-8x7b-32768"       -- long context window (32k tokens)
#   "gemma2-9b-it"             -- Google Gemma 2

GROQ_MODEL = "llama-3.1-8b-instant"  # change this to switch models

llm = ChatGroq(model=GROQ_MODEL, temperature=0)


def safe_invoke(chain, inputs, delay=5, retries=3):
    """Invoke a LangChain chain with automatic retry on rate-limit errors (HTTP 429)."""
    for attempt in range(1, retries + 1):
        try:
            return chain.invoke(inputs)
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower() or "rate limit" in str(e).lower():
                wait = delay * attempt
                print(f"Rate limit hit (attempt {attempt}/{retries}). Retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"Chain invocation failed after {retries} retries.")


print(f"Embedding model and LLM ({GROQ_MODEL}) initialised.")
print("To use a higher-quality model, set GROQ_MODEL = 'llama-3.3-70b-versatile'")

Loading embedding model (approx. 90MB, downloaded once)...


/tmp/ipykernel_201/1114452581.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model and LLM (llama-3.1-8b-instant) initialised.
To use a higher-quality model, set GROQ_MODEL = 'llama-3.3-70b-versatile'


---
## Part 1: Document Loading

Two sources are loaded:
- **Wikipedia articles** (Machine Learning, Deep Learning, NLP, Large Language Models)
- **"Attention Is All You Need"** — the original Transformer paper from arXiv

In [4]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://en.wikipedia.org/wiki/Machine_learning"
loader = WebBaseLoader(url)
web_docs = loader.load()

print(f"Documents loaded: {len(web_docs)}")
print(f"Total characters: {len(web_docs[0].page_content):,}")
print("\n--- First 500 characters ---")
print(web_docs[0].page_content[:500])
print("Metadata:", web_docs[0].metadata)

Documents loaded: 1
Total characters: 124,846

--- First 500 characters ---




Machine learning - Wikipedia



























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages



















Search











Search






















Appearance
















Donate

Create account

Log in








Personal tools





Donate Create account Log in






Metadata: {'source': 'https://en.wikipedia.org/wiki/Machine_learning', 'title': 'Machine learning - Wikipedia', 'language': 'en'}


In [5]:
# Load four Wikipedia articles to build a broader knowledge base
urls = [
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://en.wikipedia.org/wiki/Deep_learning",
    "https://en.wikipedia.org/wiki/Natural_language_processing",
    "https://en.wikipedia.org/wiki/Large_language_model",
]

loader = WebBaseLoader(urls)
all_docs = loader.load()

print(f"Documents loaded: {len(all_docs)}")
for doc in all_docs:
    print(f"  {doc.metadata.get('source', 'unknown')} -- {len(doc.page_content):,} chars")

Documents loaded: 4
  https://en.wikipedia.org/wiki/Machine_learning -- 124,846 chars
  https://en.wikipedia.org/wiki/Deep_learning -- 137,316 chars
  https://en.wikipedia.org/wiki/Natural_language_processing -- 54,894 chars
  https://en.wikipedia.org/wiki/Large_language_model -- 130,109 chars


In [6]:
import urllib.request
from langchain_community.document_loaders import PyPDFLoader

pdf_url = "https://arxiv.org/pdf/1706.03762"
pdf_path = "/tmp/attention_is_all_you_need.pdf"

print("Downloading Transformer paper PDF...")
urllib.request.urlretrieve(pdf_url, pdf_path)

pdf_loader = PyPDFLoader(pdf_path)
pdf_pages = pdf_loader.load()

print(f"Pages loaded: {len(pdf_pages)}")
print("\n--- Page 1 preview ---")
print(pdf_pages[0].page_content[:600])
print("Page metadata:", pdf_pages[0].metadata)

Pages loaded: 15

--- Page 1 preview ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstrac
Page metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'sour

---
## Part 2: Document Splitting

Documents are split into overlapping chunks before embedding. Two parameters control this:

- `chunk_size`: maximum characters per chunk
- `chunk_overlap`: number of characters shared between adjacent chunks (preserves context at boundaries)

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20,
    separators=["\n\n", "\n", ". ", " ", ""]
)

sample_text = """Machine learning (ML) is a field of study in artificial intelligence concerned
with the development of statistical algorithms that can learn from data.

ML programs are used for natural language processing, computer vision, speech recognition, and more.

Deep learning is a subset of machine learning that uses multi-layered neural networks."""

chunks = r_splitter.split_text(sample_text)
print(f"Split into {len(chunks)} chunks:\n")
for i, chunk in enumerate(chunks):
    print(f"[Chunk {i+1}] ({len(chunk)} chars): {chunk}")
    print()

Split into 4 chunks:

[Chunk 1] (78 chars): Machine learning (ML) is a field of study in artificial intelligence concerned

[Chunk 2] (72 chars): with the development of statistical algorithms that can learn from data.

[Chunk 3] (100 chars): ML programs are used for natural language processing, computer vision, speech recognition, and more.

[Chunk 4] (86 chars): Deep learning is a subset of machine learning that uses multi-layered neural networks.



In [8]:
# Compare RecursiveCharacterTextSplitter vs CharacterTextSplitter
c_splitter = CharacterTextSplitter(chunk_size=150, chunk_overlap=20, separator=' ')
print(f"RecursiveCharacterTextSplitter : {len(r_splitter.split_text(sample_text))} chunks")
print(f"CharacterTextSplitter (space)  : {len(c_splitter.split_text(sample_text))} chunks")
print("\nThe recursive splitter tries paragraph, sentence, then word boundaries before splitting mid-word.")

RecursiveCharacterTextSplitter : 4 chunks
CharacterTextSplitter (space)  : 3 chunks

The recursive splitter tries paragraph, sentence, then word boundaries before splitting mid-word.


In [9]:
# Markdown-aware splitting preserves header hierarchy as chunk metadata
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_doc = """# Machine Learning

## Supervised Learning
Uses labelled training data. The model learns a mapping from inputs to outputs.

## Unsupervised Learning
Finds patterns in unlabelled data, such as clustering or dimensionality reduction.

### Clustering
K-means and DBSCAN are popular clustering algorithms."""

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "H1"), ("##", "H2"), ("###", "H3")]
)
md_splits = md_splitter.split_text(markdown_doc)

for s in md_splits:
    print(f"Metadata: {s.metadata}")
    print(f"Content:  {s.page_content[:100]}")
    print()

Metadata: {'H1': 'Machine Learning', 'H2': 'Supervised Learning'}
Content:  Uses labelled training data. The model learns a mapping from inputs to outputs.

Metadata: {'H1': 'Machine Learning', 'H2': 'Unsupervised Learning'}
Content:  Finds patterns in unlabelled data, such as clustering or dimensionality reduction.

Metadata: {'H1': 'Machine Learning', 'H2': 'Unsupervised Learning', 'H3': 'Clustering'}
Content:  K-means and DBSCAN are popular clustering algorithms.



In [10]:
# Split all loaded documents into 1000-character chunks with 150-character overlap
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

web_splits = text_splitter.split_documents(all_docs)
pdf_splits = text_splitter.split_documents(pdf_pages)
all_splits = web_splits + pdf_splits

print(f"Web  : {len(all_docs)} documents -> {len(web_splits)} chunks")
print(f"PDF  : {len(pdf_pages)} pages -> {len(pdf_splits)} chunks")
print(f"Total: {len(all_splits)} chunks")

sample_chunk = all_splits[10]
print("\nSample chunk content:", sample_chunk.page_content[:300])
print("\nMetadata:", sample_chunk.metadata)

Web  : 4 documents -> 617 chunks
PDF  : 15 pages -> 49 chunks
Total: 666 chunks

Sample chunk content: From a theoretical viewpoint, probably approximately correct learning provides a mathematical and statistical framework for describing machine learning. Most traditional machine learning and deep learning algorithms can be described as empirical risk minimisation under this framework.

Metadata: {'source': 'https://en.wikipedia.org/wiki/Machine_learning', 'title': 'Machine learning - Wikipedia', 'language': 'en'}


---
## Part 3: Vector Stores and Embeddings

Chunks are converted to dense vector representations using `all-MiniLM-L6-v2`, a lightweight HuggingFace model that runs locally on CPU with no API calls.

### 3a. Embedding similarity

In [11]:
s1 = "machine learning is a subset of artificial intelligence"
s2 = "ML is part of the broader field of AI"
s3 = "the stock market closed higher on Friday"

e1 = embedding.embed_query(s1)
e2 = embedding.embed_query(s2)
e3 = embedding.embed_query(s3)

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"Embedding dimensions: {len(e1)}")
print(f"\nSimilarity scores (1.0 = identical meaning):")
print(f"  s1 vs s2 (same topic, different wording): {cosine_sim(e1, e2):.4f}")
print(f"  s1 vs s3 (unrelated topic):               {cosine_sim(e1, e3):.4f}")

Embedding dimensions: 384

Similarity scores (1.0 = identical meaning):
  s1 vs s2 (same topic, different wording): 0.6624
  s1 vs s3 (unrelated topic):               -0.0010


### 3b. Building the ChromaDB vector store

In [12]:
persist_directory = '/content/chroma_db'  # /content is always writable in Colab
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)

print("Embedding all chunks locally. This takes 1-2 minutes on Colab CPU...")

vectordb = Chroma.from_documents(
    documents=all_splits,
    embedding=embedding,
    persist_directory=persist_directory
)

print(f"Vector store ready. Vectors indexed: {vectordb._collection.count()}")

Embedding all chunks locally. This takes 1-2 minutes on Colab CPU...
Vector store ready. Vectors indexed: 666


In [13]:
question = "What is the difference between supervised and unsupervised learning?"
docs = vectordb.similarity_search(question, k=3)

print(f"Top {len(docs)} results:\n")
for i, doc in enumerate(docs):
    print(f"--- Result {i+1} (source: ...{str(doc.metadata.get('source','?'))[-50:]}) ---")
    print(doc.page_content[:300])
    print()

Top 3 results:

--- Result 1 (source: ...https://en.wikipedia.org/wiki/Machine_learning) ---
Feature learning can be either supervised or unsupervised. In supervised feature learning, features are learned using labelled input data. Examples include artificial neural networks, multilayer perceptrons, and supervised dictionary learning. In unsupervised feature learning, features are learned w

--- Result 2 (source: ...https://en.wikipedia.org/wiki/Machine_learning) ---
Approaches[edit]


In supervised learning, the training data is labelled with the expected answers, while in unsupervised learning, the model identifies patterns or structures in unlabelled data.
Machine learning approaches are traditionally divided into three broad categories, which correspond to l

--- Result 3 (source: ...https://en.wikipedia.org/wiki/Machine_learning) ---
Unsupervised learning[edit]
Main article: Unsupervised learningSee also: Cluster analysis
Unsupervised learning algorithms find structures in data th

### 3c. Maximum Marginal Relevance (MMR)

Standard similarity search can return near-duplicate chunks. MMR balances relevance with diversity to return a broader set of results.

In [14]:
question = "How do neural networks learn?"

docs_ss  = vectordb.similarity_search(question, k=3)
docs_mmr = vectordb.max_marginal_relevance_search(question, k=3, fetch_k=6)

print("--- Similarity Search ---")
for i, d in enumerate(docs_ss):
    print(f"[{i+1}] {d.page_content[:130]}")

print("\n--- MMR (diverse results) ---")
for i, d in enumerate(docs_mmr):
    print(f"[{i+1}] {d.page_content[:130]}")

--- Similarity Search ---
[1] Artificial neural networks[edit]
Main article: Artificial neural networkSee also: Deep learning
An artificial neural network is an
[2] Deep learning is closely related to a class of theories of brain development (specifically, neocortical development) proposed by c
[3] DNNs are typically feedforward networks in which data flows from the input layer to the output layer without looping back. At firs

--- MMR (diverse results) ---
[1] Artificial neural networks[edit]
Main article: Artificial neural networkSee also: Deep learning
An artificial neural network is an
[2] Deep learning is closely related to a class of theories of brain development (specifically, neocortical development) proposed by c
[3] DNNs are typically feedforward networks in which data flows from the input layer to the output layer without looping back. At firs


---
## Part 4: Advanced Retrieval

### 4a. Metadata filtering

In [15]:
question = "What are transformers used for?"

docs_all = vectordb.similarity_search(question, k=3)
print("Without filter:")
for d in docs_all:
    print(f"  source: ...{str(d.metadata.get('source','?'))[-55:]}")

print()
docs_pdf = vectordb.similarity_search(question, k=3, filter={"source": pdf_path})
print("PDF source only:")
for d in docs_pdf:
    print(f"  page {d.metadata.get('page','?')}: {d.page_content[:100]}")

Without filter:
  source: ...https://en.wikipedia.org/wiki/Large_language_model
  source: ...https://en.wikipedia.org/wiki/Large_language_model
  source: ...https://en.wikipedia.org/wiki/Large_language_model

PDF source only:
  page 1: The Transformer allows for significantly more parallelization and can reach a new state of the art i
  page 2: Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture us
  page 0: large and limited training data.
∗Equal contribution. Listing order is random. Jakob proposed replac


### 4b. LLM-guided source routing

The LLM classifies each question and selects the appropriate source to search. This achieves the same goal as `SelfQueryRetriever` without the brittle dependency on query constructor parsers.

In [16]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

filter_prompt = PromptTemplate.from_template("""
You are a metadata filter extractor. Given a user question, decide which source to search.

Available sources:
- "wikipedia" (articles on Machine Learning, Deep Learning, NLP, Large Language Models)
- "pdf" (the Attention Is All You Need Transformer research paper)
- "all" (search everything)

User question: {question}

Reply with ONLY one word: wikipedia, pdf, or all.
""")

filter_chain = filter_prompt | llm | StrOutputParser()

def smart_retrieve(question, k=3):
    source_filter = safe_invoke(filter_chain, {"question": question}).strip().lower()
    print(f"  Source selected: '{source_filter}'")

    if "pdf" in source_filter:
        docs = vectordb.similarity_search(question, k=k, filter={"source": pdf_path})
        print("  Searching PDF only")
    elif "wikipedia" in source_filter:
        docs = vectordb.similarity_search(question, k=k+2)
        docs = [d for d in docs if 'wikipedia' in str(d.metadata.get('source', ''))][:k]
        print("  Searching Wikipedia only")
    else:
        docs = vectordb.similarity_search(question, k=k)
        print("  Searching all sources")

    return docs

questions = [
    "What does the transformer paper say about multi-head attention?",
    "What does Wikipedia say about supervised learning?",
    "What is backpropagation?",
]

for q in questions:
    print(f"\nQ: {q}")
    results = smart_retrieve(q, k=2)
    for d in results:
        src = str(d.metadata.get('source', '?'))[-45:]
        page = d.metadata.get('page', '-')
        print(f"  [p.{page} | ...{src}] {d.page_content[:120]}")
    time.sleep(4)


Q: What does the transformer paper say about multi-head attention?
  Source selected: 'pdf'
  Searching PDF only
  [p.4 | .../tmp/attention_is_all_you_need.pdf] output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.

  [p.2 | .../tmp/attention_is_all_you_need.pdf] Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-att

Q: What does Wikipedia say about supervised learning?
  Source selected: 'wikipedia'
  Searching Wikipedia only
  [p.- | ...ttps://en.wikipedia.org/wiki/Machine_learning] Supervised learning[edit]
Main article: Supervised learning
A support-vector machine is a supervised learning model that
  [p.- | ...ttps://en.wikipedia.org/wiki/Machine_learning] Supervised learning algorithms build a mathematical model of a set of data that contains both the inputs and the desired

Q: What is backpropagation?
  Source selected: 'wikipedia'
  Searching Wik

### 4c. Contextual Compression

Each retrieved chunk is passed through an LLM extractor that returns only the portion relevant to the query, reducing noise in the context sent to the final LLM.

In [17]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

compression_retriever = ContextualCompressionRetriever(
    base_compressor=LLMChainExtractor.from_llm(llm),
    base_retriever=vectordb.as_retriever(search_type="mmr")
)

compressed_docs = compression_retriever.invoke(
    "What are the main types of machine learning?"
)

print(f"Retrieved {len(compressed_docs)} compressed chunk(s):\n")
for i, doc in enumerate(compressed_docs):
    print(f"--- Doc {i+1} ---\n{doc.page_content[:400]}\n")

Retrieved 4 compressed chunk(s):

--- Doc 1 ---
Approaches[edit]

In supervised learning, the training data is labelled with the expected answers, while in unsupervised learning, the model identifies patterns or structures in unlabelled data.
Machine learning approaches are traditionally divided into three broad categories, which correspond to learning paradigms, depending on the nature of the "signal" or "feedback" available to the learning sy

--- Doc 2 ---
Generalization[edit]
Characterizing the generalisation of various learning algorithms is an active topic of current research, especially for deep learning algorithms.

--- Doc 3 ---
Software suites containing a variety of machine learning algorithms include the following:

Free and open-source software[edit]
See also: Lists of open-source artificial intelligence software

Caffe
Deeplearning4j
DeepSpeed
ELKI
Google JAX
Infer.NET
JASP
Jubatus
Keras
Kubeflow
LightGBM
Mahout
Mallet
Microsoft Cognitive Toolkit
ML.NET
mlpack
MXNet
OpenN

---
## Part 5: Question Answering

### 5a. Basic RetrievalQA chain

In [18]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(llm, retriever=vectordb.as_retriever())
result = safe_invoke(qa_chain, {"query": "What is the attention mechanism in the Transformer model?"})
print(result["result"])

The attention mechanism in the Transformer model is a multi-head self-attention mechanism. It allows the model to jointly attend to information from different representation subspaces at different positions. This is achieved by projecting the input sequence into multiple subspaces and then computing attention weights for each subspace.

The multi-head attention mechanism is defined as follows:

MultiHead(Q, K, V) = Concat(head1, ..., headh)WO

where headi = Attention(QWQi, KWKi, V WV i)

Here, Q, K, and V are the query, key, and value matrices, respectively. WQi, WK i, and WV i are the projection matrices for each head, and WO is the output matrix.

The attention mechanism is used in three different ways in the Transformer model:

1. Encoder-decoder attention: The queries come from the previous decoder layer, and the keys and values come from the encoder output.
2. Self-attention: The queries, keys, and values come from the same input sequence.
3. Encoder self-attention: The queries, k

### 5b. Custom prompt template

In [19]:
from langchain_core.prompts import PromptTemplate

template = """You are a helpful assistant. Answer using only the context below.
If the answer is not in the context, say so. Keep answers to 3 sentences maximum.

Context:
{context}

Question: {question}
Answer:"""

qa_chain_custom = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": PromptTemplate.from_template(template)}
)

result = qa_chain_custom.invoke({"query": "How does backpropagation work in neural networks?"})
print("Answer:", result["result"])
print("\nSource documents:")
for doc in result["source_documents"]:
    print(f"  ...{str(doc.metadata.get('source','?'))[-60:]}")

Answer: Backpropagation is an efficient application of the chain rule to networks of differentiable nodes, derived from Gottfried Wilhelm Leibniz's work in 1673. It involves passing information in the reverse direction and adjusting the network to reflect that information, allowing the algorithm to make certain parameters more influential. This process helps the network to determine the correct mathematical manipulation to fully process the data.

Source documents:
  ...https://en.wikipedia.org/wiki/Deep_learning
  ...https://en.wikipedia.org/wiki/Deep_learning
  ...https://en.wikipedia.org/wiki/Deep_learning
  ...https://en.wikipedia.org/wiki/Deep_learning


### 5c. Chain types

| Chain | Behaviour |
|---|---|
| `stuff` | All retrieved chunks placed in a single prompt (fast, default) |
| `map_reduce` | Each chunk summarised individually, then combined into a final answer |
| `refine` | Answer iteratively refined as each chunk is processed |

In [20]:
question = "What are the applications of large language models?"

for chain_type in ["stuff", "map_reduce"]:
    qa = RetrievalQA.from_chain_type(llm, retriever=vectordb.as_retriever(), chain_type=chain_type)
    result = safe_invoke(qa, {"query": question})
    print(f"--- {chain_type.upper()} ---")
    print(result["result"])
    print()
    time.sleep(4)

--- STUFF ---
Large language models have a wide range of applications across various industries and domains. Some of the key applications include:

1. **Natural Language Processing (NLP)**: Large language models can be used for tasks such as language translation, text summarization, sentiment analysis, and question answering.
2. **Chatbots and Virtual Assistants**: Large language models can be used to power chatbots and virtual assistants, enabling them to understand and respond to user queries in a more human-like way.
3. **Text Generation**: Large language models can be used to generate text, such as articles, stories, and even entire books.
4. **Language Translation**: Large language models can be used to translate text from one language to another, enabling communication across language barriers.
5. **Sentiment Analysis**: Large language models can be used to analyze text and determine the sentiment or emotional tone behind it.
6. **Question Answering**: Large language models can b

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

--- MAP_REDUCE ---
Based on general knowledge, large language models have a wide range of applications, including:

- Text generation and completion
- Language translation
- Sentiment analysis and opinion mining
- Question answering and conversational AI
- Summarization and abstracting
- Content creation and writing assistance
- Chatbots and virtual assistants
- Language learning and education
- Text classification and categorization
- Named entity recognition and extraction
- Part-of-speech tagging and dependency parsing

Additionally, some other applications of large language models include:

- Text generation: generating human-like text for various purposes such as chatbots, language translation, and content creation.
- Sentiment analysis: analyzing text to determine the sentiment or emotional tone behind it.
- Question answering: answering questions based on the input text or knowledge base.
- Language translation: translating text from one language to another.
- Summarization: sum

### 5d. The stateless problem

`RetrievalQA` does not retain conversation history. Each call is independent, so follow-up questions that reference earlier answers will fail.

In [21]:
qa = RetrievalQA.from_chain_type(llm, retriever=vectordb.as_retriever())

r1 = safe_invoke(qa, {"query": "What is supervised learning?"})
print("Q1: What is supervised learning?\nA1:", r1["result"])

time.sleep(4)

r2 = safe_invoke(qa, {"query": "Can you give me an example of that?"})
print("\nQ2: Can you give me an example of that?\nA2:", r2["result"])
print("\nNote: 'that' has no referent -- the chain has no memory of Q1. Addressed in Part 6.")

Q1: What is supervised learning?
A1: Supervised learning is a type of machine learning where the computer is presented with example inputs and their desired outputs, given by a "teacher". The goal is to learn a general rule that maps inputs to outputs. This type of learning involves training a model on labeled data, where the correct answers are already known, and the model learns to make predictions or classify new, unseen data.

Q2: Can you give me an example of that?
A2: I can give you an example of a "hallucination" in the context of natural language processing. 

For instance, if a model is trained on a dataset that includes information about the voting process in the United States, it might generate a sentence like: "The voting age in the United States is 21." This sentence appears to be factually sound, but it's actually incorrect - the voting age in the United States is actually 18.

This type of error is an example of a hallucination, where the model generates text that is syn

---
## Part 6: Conversational Chat with Memory

`ConversationalRetrievalChain` stores the conversation history and rephrases follow-up questions in context before passing them to the retriever.

### 6a. Basic conversational chain

In [22]:
from langchain_classic.chains.conversational_retrieval.base import ConversationalRetrievalChain
from langchain_classic.memory.buffer import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

qa_conv = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=vectordb.as_retriever(),
    memory=memory
)

for q in [
    "What is supervised learning?",
    "Can you give me a concrete example of that?",
    "How does that compare to unsupervised learning?"
]:
    result = safe_invoke(qa_conv, {"question": q})
    print(f"Q: {q}\nA: {result['answer']}\n")
    time.sleep(4)

/tmp/ipykernel_201/559468887.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


Q: What is supervised learning?
A: Supervised learning is a type of machine learning where the computer is presented with example inputs and their desired outputs, given by a "teacher". The goal is to learn a general rule that maps inputs to outputs. This type of learning involves training a model on labeled data, where the correct answers are already known, and the model learns to make predictions or classify new, unseen data.

Q: Can you give me a concrete example of that?
A: A classic example of supervised learning is a spam email filter. 

Here's how it works:

1. A company collects a large dataset of emails, labeled as either "spam" or "not spam" by a human.
2. The dataset is used to train a machine learning model, which learns to recognize patterns in the emails that are indicative of spam.
3. The model is trained on the labeled data, and it learns to identify features such as certain keywords, phrases, and formatting that are commonly found in spam emails.
4. Once the model is t

In [23]:
print("Conversation history:")
for msg in memory.chat_memory.messages:
    role = "User" if msg.type == "human" else "Assistant"
    print(f"  [{role}]: {str(msg.content)[:100]}")

Conversation history:
  [User]: What is supervised learning?
  [Assistant]: Supervised learning is a type of machine learning where the computer is presented with example input
  [User]: Can you give me a concrete example of that?
  [Assistant]: A classic example of supervised learning is a spam email filter. 

Here's how it works:

1. A compan
  [User]: How does that compare to unsupervised learning?
  [Assistant]: Supervised learning and unsupervised learning are two main categories of machine learning approaches


### 6b. Chatbot with custom prompt and MMR retrieval

In [24]:
def create_chatbot(vectordb, llm):
    memory = ConversationBufferMemory(
        memory_key="chat_history", return_messages=True, output_key='answer'
    )
    prompt = PromptTemplate.from_template(
        """You are a knowledgeable assistant. Answer using the context provided.
If the answer is not in the context, say so. Be concise.

Context: {context}
Question: {question}
Answer:"""
    )
    return ConversationalRetrievalChain.from_llm(
        llm,
        retriever=vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 4}),
        memory=memory,
        return_source_documents=True,
        combine_docs_chain_kwargs={"prompt": prompt}
    )

def chat(chain, question):
    result = chain.invoke({"question": question})
    print(f"\nUser: {question}")
    print(f"\nAssistant: {result['answer']}")
    sources = set(str(d.metadata.get('source',''))[-50:] for d in result.get('source_documents', []))
    if sources:
        print(f"\nSources: {', '.join(sources)}")
    print("-" * 60)

print("Helper functions defined.")

Helper functions defined.


In [25]:
chatbot = create_chatbot(vectordb, llm)
print("Chatbot ready. Knowledge base covers: machine learning, deep learning, NLP, Transformers.")

Chatbot ready. Knowledge base covers: machine learning, deep learning, NLP, Transformers.


In [26]:
chat(chatbot, "What is deep learning and how is it different from traditional machine learning?")
time.sleep(4)


User: What is deep learning and how is it different from traditional machine learning?

Assistant: Deep learning is a subset of machine learning that focuses on utilizing multilayered neural networks to perform tasks such as classification, regression, and representation learning. It differs from traditional machine learning in that it uses multiple layers (ranging from three to several hundred or thousands) in the network, whereas traditional machine learning typically uses a single layer or a few layers.

Sources: https://en.wikipedia.org/wiki/Machine_learning, https://en.wikipedia.org/wiki/Deep_learning
------------------------------------------------------------


In [27]:
chat(chatbot, "What kinds of architectures are commonly used in it?")
time.sleep(4)


User: What kinds of architectures are commonly used in it?

Assistant: Fully connected networks, deep belief networks, recurrent neural networks, convolutional neural networks, generative adversarial networks, transformers, and neural radiance fields.

Sources: https://en.wikipedia.org/wiki/Machine_learning, https://en.wikipedia.org/wiki/Deep_learning
------------------------------------------------------------


In [28]:
chat(chatbot, "How do Transformers improve on those architectures?")
time.sleep(4)


User: How do Transformers improve on those architectures?

Assistant: Transformers improve on those architectures by reducing the number of operations required to relate signals from two arbitrary input or output positions to a constant number, regardless of their distance from each other.

Sources: /tmp/attention_is_all_you_need.pdf, https://en.wikipedia.org/wiki/Large_language_model
------------------------------------------------------------


In [29]:
chat(chatbot, "What are large language models and how do they relate to Transformers?")
time.sleep(4)


User: What are large language models and how do they relate to Transformers?

Assistant: Large language models are a type of artificial intelligence (AI) model that use neural networks to learn and generate human-like language. They were developed to move beyond traditional n-gram models and have been used for various language tasks such as translation, text generation, and language understanding.

Transformers are a specific type of neural network architecture that was introduced in 2017 and has been widely used in large language models. They are particularly well-suited for sequence-to-sequence tasks, such as machine translation, and have been shown to outperform traditional recurrent neural networks (RNNs) like LSTMs in many cases. The Transformer architecture is characterized by its use of self-attention mechanisms, which allow the model to weigh the importance of different input elements when generating output.

Sources: /tmp/attention_is_all_you_need.pdf, https://en.wikipedia.or

In [30]:
chat(chatbot, "What are some real-world applications of NLP?")
time.sleep(4)


User: What are some real-world applications of NLP?

Assistant: Some of the NLP tasks mentioned have direct real-world applications, but the text does not explicitly list them. However, it can be inferred that machine translation, which is mentioned as a notable early success in statistical methods in NLP, has real-world applications such as:

- Translating governmental proceedings into official languages
- Facilitating communication across languages in international settings

Other potential real-world applications of NLP tasks mentioned in the text include:

- Chatterbots (e.g., Racter and Jabberwacky) for customer service or entertainment
- Sentiment analysis for market research or customer feedback
- Named entity recognition for information extraction or data mining
- Text summarization for news aggregation or research assistance

However, these are not explicitly mentioned in the provided context.

Sources: /en.wikipedia.org/wiki/Natural_language_processing
----------------------

### 6c. Chat with an uploaded PDF

In [31]:
from google.colab import files
from langchain_community.vectorstores import DocArrayInMemorySearch

def chat_with_your_pdf():
    print("Upload a PDF:")
    uploaded = files.upload()
    if not uploaded:
        return None

    filename = list(uploaded.keys())[0]
    loader = PyPDFLoader(filename)
    splits = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150).split_documents(loader.load())
    print(f"{filename} loaded and split into {len(splits)} chunks.")

    db = DocArrayInMemorySearch.from_documents(splits, embedding)
    memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key='answer')
    chain = ConversationalRetrievalChain.from_llm(
        llm, retriever=db.as_retriever(search_kwargs={"k": 4}),
        memory=memory, return_source_documents=True
    )
    print(f"Ready. Ask questions about '{filename}'.")
    return chain

my_chatbot = chat_with_your_pdf()
chat(my_chatbot, "What is this document about?")

Upload a PDF:


Saving neural_networks_guide.pdf to neural_networks_guide.pdf
neural_networks_guide.pdf loaded and split into 6 chunks.
Ready. Ask questions about 'neural_networks_guide.pdf'.

User: What is this document about?

Assistant: This document appears to be an introduction to the fundamentals of deep learning, specifically neural networks. It covers the basics of neural networks, including their structure, how they learn, and different types of neural networks. It seems to be a beginner's guide to understanding the concepts and architecture of neural networks.

Sources: neural_networks_guide.pdf
------------------------------------------------------------


---
## Part 7: Alternative Stack — FAISS and Nomic Embeddings

This section demonstrates an alternative retrieval pipeline using FAISS (an in-memory vector store) and the `nomic-embed-text-v1.5` embedding model, which offers higher retrieval accuracy than `all-MiniLM-L6-v2`. The same Groq LLM from the setup cell is reused.

| Component | Parts 1-6 | Part 7 |
|-----------|-----------|--------|
| Vector Store | ChromaDB (persistent) | FAISS (in-memory) |
| Embeddings | all-MiniLM-L6-v2 | nomic-embed-text-v1.5 |
| LLM | Groq | Groq (same instance) |
| Source | Wikipedia + arXiv PDF | Uploaded PDF |

In [32]:
# The Groq LLM initialised in the setup cell is reused here.
print("Using existing LLM:", GROQ_MODEL)

Using existing LLM: llama-3.1-8b-instant


In [33]:
from langchain_huggingface import HuggingFaceEmbeddings as HFEmbeddings
from langchain_community.vectorstores import FAISS

# nomic-embed-text-v1.5 produces higher quality embeddings than all-MiniLM-L6-v2
nomic_embedding = HFEmbeddings(
    model_name="nomic-ai/nomic-embed-text-v1.5",
    model_kwargs={"trust_remote_code": True}
)

print("Nomic embedding model ready.")

modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Nomic embedding model ready.


### 7a. Upload and index a PDF

In [34]:
from google.colab import files as colab_files

print("Upload a PDF:")
uploaded = colab_files.upload()

if uploaded:
    pdf_filename = list(uploaded.keys())[0]
    print(f"Uploaded: {pdf_filename}")

    user_loader = PyPDFLoader(pdf_filename)
    user_documents = user_loader.load()
    print(f"Pages loaded: {len(user_documents)}")

    user_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    user_texts = user_splitter.split_documents(user_documents)

    for idx, t in enumerate(user_texts):
        t.metadata["id"] = idx

    print(f"Chunks created: {len(user_texts)}")
    print("\nSample chunk:")
    print(user_texts[0].page_content[:300])
else:
    print("No file uploaded.")

Upload a PDF:


Saving neural_networks_guide.pdf to neural_networks_guide (1).pdf
Uploaded: neural_networks_guide (1).pdf
Pages loaded: 3
Chunks created: 12

Sample chunk:
Introduction to Neural Networks
 A Beginner's Guide to Deep Learning Fundamentals
1. What is a Neural Network?
A neural network is a machine learning model inspired by the structure of the human brain. It is
composed of layers of interconnected nodes, called neurons, that process and transform data.


### 7b. Build FAISS index and run Q&A

In [35]:
if uploaded:
    print("Building FAISS index with Nomic embeddings...")
    faiss_retriever = FAISS.from_documents(user_texts, nomic_embedding).as_retriever(
        search_kwargs={"k": 5}
    )

    def pretty_print_docs(docs):
        for i, d in enumerate(docs):
            print(f"{'─' * 50}\nDocument {i+1}:")
            print(f"Content:\n{d.page_content}\n")
            for key, value in d.metadata.items():
                print(f"  {key}: {value}")
        print("─" * 50)

    groq_qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=faiss_retriever)

    query = "Give me a summary of this document."
    response = groq_qa_chain.invoke(query)
    print("\nAnswer:")
    print(response["result"])
else:
    print("Upload a PDF in the cell above first.")

Building FAISS index with Nomic embeddings...

Answer:
This document appears to be a sample for RAG (Retrieval-Augmented Generation) testing, focusing on the topic of neural networks and their applications. It provides an overview of the following key points:

1. **Neural Networks**: A brief explanation of how neural networks work, including the concept of neurons receiving inputs, applying mathematical transformations, and passing results to the next layer.
2. **Types of Neural Networks**:
	* **Feedforward Neural Networks (FNN)**: The simplest type of neural network, where data flows in one direction from input to output.
	* **Convolutional Neural Networks (CNN)**: Designed for grid-like data such as images, CNNs use convolutional layers to detect spatial features like edges, textures, and shapes.
	* **Recurrent Neural Networks (RNN)**: Built for sequential data such as text, speech, and time series, RNNs have connections that loop back on themselves, allowing information to persist a

In [36]:
if uploaded:
    my_query = "What are the main conclusions of this document?"  # change this line
    response = groq_qa_chain.invoke(my_query)
    print(f"Q: {my_query}\n")
    print("A:", response["result"])

Q: What are the main conclusions of this document?

A: Based on the provided context, I don't see any explicit conclusions. The document appears to be a sample for RAG (Retrieval-Augmented Generation) testing and provides an overview of various topics related to neural networks and their applications, such as image recognition, natural language processing, and autonomous vehicles.

However, some key points that can be inferred from the document are:

1. Neural networks are a powerful tool for various tasks, including image recognition, language understanding, and decision-making.
2. Different types of neural networks, such as Feedforward Neural Networks (FNN), Recurrent Neural Networks (RNN), and Transformer Networks, are suited for different types of data and tasks.
3. These neural networks have been successfully applied in various domains, including computer vision, natural language processing, speech recognition, recommendation systems, medical diagnosis, and autonomous vehicles.

O